In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch, Rectangle

warnings.filterwarnings("ignore")

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42

Path("figures").mkdir(parents=True, exist_ok=True)


In [ ]:
major_color_map = {
    "CD8_T": "#9edae5",
    "CD4_T": "#ff9896",
    "GD_T":  "#ffff99",
    "MAI_T": "#c5b0d5",
    "Pro_T": "#fdd0a2",
    "IM_T":  "#98df8a",
}


In [ ]:
meta_color_map = {
    "Child": "#fdae6b",
    "Adult": "#9ecae1",
    'T': '#ffc89d',
    'B': '#f2a2af',
    '10X5': '#57CBE1',
    '10X3': '#B8DFC6',
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}


In [ ]:
T = sc.read_h5ad( 'T.h5ad')

In [ ]:
tcr=sc.read_h5ad('tcr.h5ad')
ir.pp.merge_with_ir(T,tcr)
ir.tl.chain_qc(T)


# Fig 1b

In [ ]:
ax = sc.pl.umap(
    T,
    color="T_sub_level_1",
    palette=major_color_map,
    use_raw=False,
    frameon=False,
    size=2,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('figures/Fig1b_umap.pdf', dpi=600, bbox_inches="tight")
plt.show()


# Fig 1c

In [ ]:
disease_order = ['pHGG','pLGG',"MB","NB","HB","WT",'PB','aHCC',"LC","BRCA","HCC",'RC','PC',"CRC"]
column_order = ['T', 'B']

cell_counts = (
    T.obs.groupby(['Disease', 'Sample_Type'])
      .size()
      .unstack(fill_value=0)
)

available = [c for c in column_order if c in cell_counts.columns]
cell_counts = cell_counts.reindex(columns=available)

missing = [c for c in column_order if c not in cell_counts.columns]
if missing:
    raise ValueError(f"数据中未找到这些 Sample_Type：{missing}")

if disease_order:
    disease_in_data = [d for d in disease_order if d in cell_counts.index]
    others = [d for d in cell_counts.index if d not in disease_in_data]
    cell_counts = cell_counts.reindex(index=disease_in_data + others)

fig = plt.figure(figsize=(4, 6))
ax = fig.add_axes([0.16, 0.10, 0.52, 0.84])

colors = [meta_color_map[c] for c in cell_counts.columns]
cell_counts.plot(
    kind='barh',
    stacked=True,
    ax=ax,
    width=0.8,
    color=colors,
    edgecolor='none'
)

ax.set_xlabel('Cell Count')
ax.set_ylabel('Disease')

ax.legend(
    title='Sample_Type',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False,
    borderaxespad=0
)

plt.savefig(
    'figures/Fig1c_sample_type_counts.pdf',
    dpi=300,
    bbox_inches='tight'
)
plt.show()


In [ ]:
disease_order = ['pHGG','pLGG',"MB","NB","HB","WT",'PB','aHCC',"LC","BRCA","HCC",'RC','PC',"CRC"]
column_order = ["10X5", "10X3"]

paired_outline_color = "#1f78a8"

T.obs["full_paired"] = (
    (T.obs["receptor_subtype"] == "TRA+TRB") &
    (T.obs["chain_pairing"].isin([
        "single pair",
        "extra VJ",
        "extra VDJ",
        "two full chains"
    ]))
)

cell_counts = (
    T.obs.groupby(["Disease", "Sequence"])
    .size()
    .unstack(fill_value=0)
)

cell_counts = cell_counts.reindex(columns=[c for c in column_order if c in cell_counts.columns])

paired_counts = (
    T.obs.loc[(T.obs["Sequence"] == "10X5") & (T.obs["full_paired"])]
    .groupby("Disease")
    .size()
    .reindex(cell_counts.index, fill_value=0)
)

if disease_order:
    disease_in_data = [d for d in disease_order if d in cell_counts.index]
    others = [d for d in cell_counts.index if d not in disease_in_data]
    new_index = disease_in_data + others
    cell_counts = cell_counts.reindex(index=new_index)
    paired_counts = paired_counts.reindex(new_index, fill_value=0)

fig = plt.figure(figsize=(4, 6))
ax = fig.add_axes([0.16, 0.10, 0.52, 0.84])

colors = [meta_color_map[c] for c in cell_counts.columns]
cell_counts.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    width=0.8,
    color=colors,
    edgecolor="none"
)

y = np.arange(len(cell_counts))
ax.barh(
    y=y,
    width=paired_counts.values,
    left=0,
    height=0.75,
    facecolor="none",
    edgecolor=paired_outline_color,
    linewidth=1.2,
    zorder=4
)

ax.set_xlabel("Cell Count")
ax.set_ylabel("Disease")

handles = [
    Patch(facecolor=meta_color_map["10X5"], edgecolor="none", label="10X5"),
    Patch(facecolor=meta_color_map["10X3"], edgecolor="none", label="10X3"),
    Patch(facecolor="none", edgecolor=paired_outline_color, linewidth=1.2, label="Paired chains"),
]

ax.legend(
    handles=handles,
    title="Sequence",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    borderaxespad=0
)

plt.savefig(
    'figures/Fig1c_assay_paired_chains.pdf',
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# Fig 1d

In [ ]:
marker_genes_dict = {
    "CD3":["CD3D","CD3G","CD3E"],
    "CD8/CD4":["CD8A","CD8B","CD4"],
    "GDT":["TRDV2","TRGV9"],
     "MAIT":["TRAV1-2","SLC4A10"],
    "Proliferative":["MKI67","TOP2A"],
    "Immature":["RAG1","DNTT"]
}

base_cmap = mpl.colormaps["coolwarm"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_coolwarm",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

available_genes = set(T.var_names)

filtered_marker_genes_dict = {
    category: [gene for gene in genes if gene in available_genes]
    for category, genes in marker_genes_dict.items()
}

filtered_marker_genes_dict = {
    k: v for k, v in filtered_marker_genes_dict.items() if v
}

if not filtered_marker_genes_dict:
    raise ValueError("没有找到可以用于绘图的基因，请检查 marker_genes_dict 和数据集中的基因名是否匹配。")

ax=sc.pl.dotplot(
    T,
    filtered_marker_genes_dict,
    'T_sub_level_1',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.savefig('figures/Fig1d_markers.pdf', bbox_inches="tight")
ax.show()


# Fig 1e

In [ ]:
T.obs['receptor_capture'] = T.obs['receptor_subtype'].astype("category")
T.obs['receptor_capture'] = T.obs['receptor_capture'].cat.add_categories(['undetected','absent'])
receptor_map = {
    "TRA+TRB": "αβTCR",
    "TRG+TRD": "undetected"
}
T.obs['receptor_capture'] = T.obs['receptor_capture'].map(receptor_map).fillna("undetected")

seq = T.obs["Sequence"].astype(str).str.strip()
sid = T.obs["Sample_ID"].astype(str).str.strip()
mask_union = (seq == "10X3") | (sid == "GSE254250_309")
T.obs.loc[mask_union, "receptor_capture"] = "absent"

if pd.api.types.is_categorical_dtype(T.obs['receptor_capture']):
   T.obs['receptor_capture'] = T.obs['receptor_capture'].cat.add_categories(['TRAV1-2'])

T.obs.loc[T.obs['IR_VJ_1_v_call'] == 'TRAV1-2', 'receptor_capture'] = 'TRAV1-2'


In [ ]:
plt.figure(figsize=(7, 6))

viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(T.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
umap_df['receptor_capture'] = T.obs['receptor_capture'].values

subset_all = umap_df
subset_undetected = umap_df[umap_df['receptor_capture'] == 'undetected']
subset_capture = umap_df[umap_df['receptor_capture'] == 'αβTCR']
subset_mait = umap_df[umap_df['receptor_capture'] == 'TRAV1-2']

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.05
)

plt.figure(figsize=(7, 6))

sns.kdeplot(
    data=subset_all,
    x='UMAP1',
    y='UMAP2',
    **kws_background
)

sns.kdeplot(
    data=subset_mait,
    x='UMAP1',
    y='UMAP2',
    **kws_foreground
)

x_min, x_max = subset_all['UMAP1'].min(), subset_all['UMAP1'].max()
y_min, y_max = subset_all['UMAP2'].min(), subset_all['UMAP2'].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

plt.xlim(x_min - x_margin, x_max + x_margin)
plt.ylim(y_min - y_margin, y_max + y_margin)

plt.tight_layout()

plt.gca().set_axis_off()
plt.savefig('figures/Fig1e_TRAV1_2_density.tiff', format="tiff", dpi=300, bbox_inches="tight",transparent=True)
plt.show()


In [ ]:
plt.figure(figsize=(7, 6))

viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(T.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
umap_df['receptor_capture'] = T.obs['receptor_capture'].values

subset_all = umap_df
subset_undetected= umap_df[umap_df['receptor_capture'] == 'undetected']
subset_capture = umap_df[umap_df['receptor_capture'] == 'αβTCR']
subset_mait = umap_df[umap_df['receptor_capture'] == 'TRAV1-2']

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.1
)

plt.figure(figsize=(7, 6))

sns.kdeplot(
    data=subset_all,
    x='UMAP1',
    y='UMAP2',
    **kws_background
)

sns.kdeplot(
    data=subset_undetected,
    x='UMAP1',
    y='UMAP2',
    **kws_foreground
)

x_min, x_max = subset_all['UMAP1'].min(), subset_all['UMAP1'].max()
y_min, y_max = subset_all['UMAP2'].min(), subset_all['UMAP2'].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

plt.xlim(x_min - x_margin, x_max + x_margin)
plt.ylim(y_min - y_margin, y_max + y_margin)

plt.tight_layout()

plt.gca().set_axis_off()
plt.savefig('figures/Fig1e_abTCR_undetected_density.tiff', format="tiff", dpi=300, bbox_inches="tight",transparent=True)
plt.show()


# Fig 1f

In [ ]:
counts = (
    T.obs["T_sub_level_1"]
    .astype("category")
    .value_counts(dropna=False)
)

colors = [
    major_color_map.get(str(k), "#CCCCCC")
    for k in counts.index
]

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    counts.values,
    colors=colors,
    startangle=90,
    counterclock=False,
)

ax.set_title("")
ax.axis("equal")
ax.axis("off")
if ax.get_legend():
    ax.legend_.remove()
plt.tight_layout(pad=0)
plt.savefig('figures/Fig1f_cell_types.pdf', dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
print(counts)


# Fig S1b

In [ ]:
file_path = 'scib_metrics.csv'
df = pd.read_csv(file_path)

methods = [
    "Unintegrated",
    "Harmony",
    "Scanorama",
    "scVI"
]

metric_order = [
    "Total",
    "Bio_conservation",
    "Batch_correction",
    "Leiden_ARI",
    "Leiden_NMI",
    "Silhouette_label",
    "cLISI",
    "Isolated_labels",
    "BRAS",
    "Graph_connectivity",
    "KBET",
    "iLISI"
]

raw = (
    df.set_index("Embedding")
      .loc[metric_order, methods]
      .T
)

raw = raw.loc[:, ::-1]

scaled = raw.copy()

display_labels = {
    "Bio_conservation": "Bio conservation",
    "Batch_correction": "Batch correction",
    "Leiden_ARI": "Leiden ARI",
    "Leiden_NMI": "Leiden NMI",
    "Silhouette_label": "Silhouette label",
    "Isolated_labels": "Isolated labels",
    "Graph_connectivity": "Graph connectivity"
}

xlabels = [display_labels.get(x, x) for x in raw.columns]

metric_type = {
    "Total": "Aggregate score",
    "Bio_conservation": "Aggregate score",
    "Batch_correction": "Aggregate score",

    "Leiden_ARI": "Bio conservation",
    "Leiden_NMI": "Bio conservation",
    "Silhouette_label": "Bio conservation",
    "cLISI": "Bio conservation",
    "Isolated_labels": "Bio conservation",

    "BRAS": "Batch correction",
    "Graph_connectivity": "Batch correction",
    "KBET": "Batch correction",
    "iLISI": "Batch correction"
}

type_colors = {
    "Aggregate score": "#ffab91",
    "Bio conservation": "#b2dfdb",
    "Batch correction": "#81d4fa"
}

base_cmap = mpl.colormaps["coolwarm"]

cmap = LinearSegmentedColormap.from_list(
    "trunc_coolwarm",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

norm = Normalize(vmin=0, vmax=1)

fig = plt.figure(figsize=(10, 5.5), dpi=300)

ax = fig.add_axes([0.18, 0.28, 0.62, 0.45])

im = ax.imshow(
    scaled.values,
    cmap=cmap,
    norm=norm,
    aspect="auto"
)

for i in range(raw.shape[0]):
    for j in range(raw.shape[1]):
        value = raw.iloc[i, j]

        text_color = (
            "white"
            if (value > 0.75 or value < 0.25)
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:.2f}".rstrip("0").rstrip("."),
            ha="center",
            va="center",
            fontsize=10,
            color=text_color
        )

ax.set_xticks(np.arange(len(xlabels)))
ax.set_xticklabels(
    xlabels,
    fontsize=11,
    rotation=45,
    ha="left",
    rotation_mode="anchor"
)

ax.xaxis.tick_top()

ax.tick_params(
    axis="x",
    which="major",
    top=True,
    bottom=False,
    length=4,
    width=0.8,
    direction="out",
    pad=8
)

ax.set_yticks(np.arange(len(methods)))
ax.set_yticklabels(methods, fontsize=12)

ax.tick_params(
    axis="y",
    which="major",
    left=True,
    right=False,
    length=5,
    width=0.8,
    direction="out",
    pad=5
)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_xticks(np.arange(-0.5, raw.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, raw.shape[0], 1), minor=True)

ax.grid(which="minor", color="white", linewidth=2)

ax.tick_params(axis="x", which="minor", length=0)
ax.tick_params(axis="y", which="minor", length=0)

for sep in [3.5, 8.5]:
    ax.axvline(
        sep,
        color="white",
        linewidth=4
    )

ax_type = fig.add_axes([0.18, 0.75, 0.62, 0.045])

ax_type.set_xlim(-0.5, raw.shape[1] - 0.5)
ax_type.set_ylim(0, 1)
ax_type.axis("off")

for i, metric in enumerate(raw.columns):
    ax_type.add_patch(
        Rectangle(
            (i - 0.5, 0),
            1,
            1,
            color=type_colors[metric_type[metric]]
        )
    )

for sep in [3.5, 8.5]:
    ax_type.axvline(
        sep,
        color="white",
        linewidth=3
    )

fig.text(
    0.82,
    0.77,
    "Type",
    fontsize=13,
    ha="center"
)

fig.text(0.18, 0.16, "Type", fontsize=16)

legend_ax = fig.add_axes([0.18, 0.05, 0.35, 0.08])
legend_ax.axis("off")

for i, (label, color) in enumerate(type_colors.items()):
    y = 0.75 - i * 0.32

    legend_ax.add_patch(
        Rectangle(
            (0, y),
            0.08,
            0.18,
            color=color,
            transform=legend_ax.transAxes
        )
    )

    legend_ax.text(
        0.12,
        y + 0.09,
        label,
        va="center",
        fontsize=11,
        transform=legend_ax.transAxes
    )

fig.text(0.60, 0.16, "Scaled score", fontsize=16)

cbar_ax = fig.add_axes([0.60, 0.08, 0.30, 0.025])

gradient = np.linspace(0, 1, 256).reshape(1, -1)

cbar_ax.imshow(
    gradient,
    aspect="auto",
    cmap=cmap,
    norm=norm,
    extent=[0, 1, 0, 1]
)

cbar_ax.set_xlim(0, 1)
cbar_ax.set_ylim(0, 1)
cbar_ax.set_yticks([])

cbar_ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
cbar_ax.set_xticklabels(
    ["0", "0.25", "0.5", "0.75", "1"],
    fontsize=10
)

cbar_ax.tick_params(
    axis="x",
    length=4,
    width=0.8,
    direction="out",
    pad=3
)

for spine in cbar_ax.spines.values():
    spine.set_visible(False)

plt.savefig(
    'figures/FigS1b_integration_benchmark.pdf',
    bbox_inches="tight"
)
plt.show()


# Fig S1c

In [ ]:
T.obs["Origin_Period_Sample"] = (
    T.obs["Origin"].map(lambda x: "inhouse" if x == "Dong_lab" else "published").astype(str)
    + "_"
    + T.obs["Period_merged"].astype(str)
    + "_"
    + T.obs["Sample_Type"].astype(str)
)
print(T.obs['Origin_Period_Sample'].value_counts())


In [ ]:
outdir = 'figures'
os.makedirs(outdir, exist_ok=True)

viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(T.obsm["X_umap"], columns=["UMAP1", "UMAP2"])
umap_df["Origin_Period_Sample"] = T.obs["Origin_Period_Sample"].values

subset_all = umap_df

groups = [
    "published_Adult_T",
    "published_Child_T",
    "inhouse_Child_B",
    "inhouse_Child_T"
]

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.02
)

x_min, x_max = subset_all["UMAP1"].min(), subset_all["UMAP1"].max()
y_min, y_max = subset_all["UMAP2"].min(), subset_all["UMAP2"].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

for g in groups:
    subset_fg = umap_df[umap_df["Origin_Period_Sample"] == g]

    plt.figure(figsize=(7, 6))

    sns.kdeplot(
        data=subset_all,
        x="UMAP1",
        y="UMAP2",
        **kws_background
    )

    sns.kdeplot(
        data=subset_fg,
        x="UMAP1",
        y="UMAP2",
        **kws_foreground
    )

    plt.xlim(x_min - x_margin, x_max + x_margin)
    plt.ylim(y_min - y_margin, y_max + y_margin)

    plt.gca().set_axis_off()
    plt.tight_layout()

    outfile = os.path.join(outdir, f"FigS1c_{g}_density.tiff")
    plt.savefig(outfile, format="tiff", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    plt.close()


# Fig S1d

In [ ]:
T.obs["Tumor_Blood"] = np.where(
    T.obs["Sample_Type"].astype(str) == "T",
    T.obs["Disease"].astype(str),
    np.where(
        T.obs["Sample_Type"].astype(str) == "B",
        "PST_Blood",
        np.nan
    )
)


In [ ]:
disease_order = [
    "PST_Blood",'pHGG','pLGG',"MB","NB","HB","WT",'PB',
    'aHCC',"LC","BRCA","HCC",'RC','PC',"CRC"
]

sublevel_order = []

df = T.obs[["Tumor_Blood", "T_sub_level_1"]].copy()
df = df.dropna(subset=["Tumor_Blood", "T_sub_level_1"])
df["Tumor_Blood"] = df["Tumor_Blood"].astype(str)
df["T_sub_level_1"] = df["T_sub_level_1"].astype(str)

prop = pd.crosstab(
    df["Tumor_Blood"],
    df["T_sub_level_1"],
    normalize="index"
)

present = [d for d in disease_order if d in prop.index]
others  = [d for d in prop.index if d not in disease_order]
prop = prop.loc[present + others]

if sublevel_order:
    cols_present = [c for c in sublevel_order if c in prop.columns]
    cols_others  = [c for c in prop.columns if c not in cols_present]
    prop = prop[cols_present + cols_others]
else:
    prop = prop[prop.sum(axis=0).sort_values(ascending=False).index]

colors = [
    major_color_map.get(c, "#CCCCCC")
    for c in prop.columns
]

ax = prop.plot(
    kind="barh",
    stacked=True,
    figsize=(5, 6),
    width=0.8,
    edgecolor="none",
    color=colors
)

ax.set_xlabel("Proportion")
ax.set_ylabel("")
ax.set_xlim(0, 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

ax.tick_params(axis="both", which="both", direction="out", width=0.8)

ax.legend(
    title="T_sub_level_1",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.savefig('figures/FigS1d_cell_type_proportions.pdf',
             dpi=300, bbox_inches="tight")
plt.show()
